In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
pip install dagshub mlflow scikit-learn pandas matplotlib seaborn skops --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import dagshub
import mlflow
import mlflow.sklearn
import skops.io as sio

dagshub.init(repo_owner='myvari', repo_name='House-Prices', mlflow=True)

Accessing as myvari

Initialized MLflow to track repo "myvari/House-Prices"

Repository myvari/House-Prices initialized!

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin

class DataCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Drop ID
        X = X.drop("Id", axis=1)
        X = X.drop("MiscFeature", axis=1)
        X = X.drop("MiscVal", axis=1)
        X = X.drop("Alley", axis=1)
        X = X.drop("Fence", axis=1)

        # Categorical NA to "None"
        cat_none_cols = [
            "PoolQC", "FireplaceQu",
            "GarageType", "GarageFinish", "GarageQual", "GarageCond",
            "BsmtQual", "BsmtCond", "BsmtExposure",
            "BsmtFinType1", "BsmtFinType2",
            "MasVnrType",
            #"MiscFeature", "Alley", "Fence"
        ]

        for col in cat_none_cols:
            if col in X:
                X[col] = X[col].fillna("None")

        # Numeric NA to 0 (for features where NA means "no feature")
        if "GarageYrBlt" in X:
            X["GarageYrBlt"] = X["GarageYrBlt"].fillna(0)

        if "MasVnrArea" in X:
            X["MasVnrArea"] = X["MasVnrArea"].fillna(0)

        # LotFrontage (Neighborhood group median)
        if "LotFrontage" in X:
            X["LotFrontage"] = X.groupby("Neighborhood")["LotFrontage"].transform(
                lambda x: x.fillna(x.median())
            )

        # Electrical (mode)
        if "Electrical" in X:
            X["Electrical"] = X["Electrical"].fillna(X["Electrical"].mode()[0])

        return X
    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

In [5]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Presence features
        X["HasGarage"] = (X["GarageType"] != "None").astype(int)
        X["HasPool"] = (X["PoolQC"] != "None").astype(int)
        X["HasMasVnr"] = (X["MasVnrType"] != "None").astype(int)
        X["HasFireplace"] = (X["FireplaceQu"] != "None").astype(int)

        # Total area
        X["TotalSF"] = X["TotalBsmtSF"] + X["1stFlrSF"] + X["2ndFlrSF"]

        return X
    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

In [6]:
class CorrelationReducer(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.8, verbose=True):
        self.threshold = threshold
        self.verbose = verbose

    def fit(self, X, y=None):
        X_df = X.copy()
        self.feature_names_ = X_df.columns.tolist()

        corr_matrix = X_df.corr().abs()

        # upper triangle
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
        upper = corr_matrix.where(mask)

        to_remove = set()

        for col in upper.columns:
            correlated = upper[col][upper[col] > self.threshold].index.tolist()

            for other in correlated:
                # overall correlation strength
                col_score = corr_matrix[col].sum()
                other_score = corr_matrix[other].sum()

                if col_score > other_score:
                    to_remove.add(col)
                else:
                    to_remove.add(other)

        self.removed_features_ = list(to_remove)
        self.kept_features_ = [f for f in self.feature_names_ if f not in to_remove]

        if self.verbose:
            print(f"[Correlation] Removed {len(self.removed_features_)} features: {self.removed_features_}")
            print(f"[Correlation] Kept {len(self.kept_features_)} features: {self.kept_features_}")

        return self

    def transform(self, X):
        X_df = X.copy()
        X_df.columns = self.feature_names_

        return X_df[self.kept_features_]
    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

In [7]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression


class FeatureSelectorRFE(BaseEstimator, TransformerMixin):
    def __init__(self, n_features=20, verbose=True):
        self.n_features = n_features
        self.verbose = verbose

    def fit(self, X, y):
        X_df = X.copy()
        self.input_features_ = X_df.columns.tolist()

        model = LinearRegression()

        self.selector_ = RFE(
            estimator=model,
            n_features_to_select=self.n_features
        )
        self.selector_.fit(X_df, y)

        mask = self.selector_.support_
        self.selected_features_ = list(pd.Index(self.input_features_)[mask])

        if self.verbose:
            print(f"[RFE] Selected {len(self.selected_features_)} features: {self.selected_features_}")

        return self

    def transform(self, X):
        X_df = X.copy()
        X_df.columns = self.input_features_

        return X_df[self.selected_features_]
    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

In [9]:
preprocessing_run_id = "fd50cf1dce8d4bedb5f5b4f06bf1ccee" 
best_run_id          = "474dc2a8677f46ca8a03e84f59547101"

# load preprocessing pipeline
trusted_types = [
    "__main__.DataCleaner",
    "__main__.FeatureEngineer",
    "__main__.CorrelationReducer",
    "__main__.FeatureSelectorRFE",
    "sklearn.pipeline.Pipeline",
    "sklearn.compose._column_transformer.ColumnTransformer",
    "sklearn.compose._column_transformer._RemainderColsList",
    "sklearn.impute._base.SimpleImputer",
    "sklearn.preprocessing._encoders.OneHotEncoder",
    "sklearn.linear_model._base.LinearRegression",
    "sklearn.tree._classes.DecisionTreeRegressor",
    "sklearn.ensemble._forest.RandomForestRegressor",
    "sklearn.feature_selection._rfe.RFE",
    "numpy.dtype",
    "pandas.core.frame.DataFrame",
    "pandas.core.series.Series",
    "xgboost.core.Booster",
    "xgboost.sklearn.XGBRegressor",
]

pipeline_path = mlflow.artifacts.download_artifacts(
    run_id=preprocessing_run_id,
    artifact_path="preprocessing_pipeline.skops"
)
preprocessing_pipeline = sio.load(pipeline_path, trusted=trusted_types)
print("Preprocessing pipeline loaded..")

# load best model
model_path = mlflow.artifacts.download_artifacts(
    run_id=best_run_id,
    artifact_path="xgb_model.skops" 
)
best_model = sio.load(model_path, trusted=trusted_types)
print("Model loaded")
print(type(best_model))

Preprocessing pipeline loaded..
Model loaded
<class 'xgboost.sklearn.XGBRegressor'>


In [11]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
best_run = client.get_run(best_run_id)

log_target_tag = best_run.data.tags.get("log_target", "False")
use_log_target = str(log_target_tag).lower() == "true"

print("log_target:", use_log_target)

log_target: True


In [33]:
test_df = pd.read_csv('data/raw/test.csv')
Ids = test_df["Id"]

X_kaggle = preprocessing_pipeline.transform(test_df)
kaggle_predictions = best_model.predict(X_kaggle)

# convert back if model was trained on log(target)
if use_log_target:
    kaggle_predictions = np.expm1(kaggle_predictions)

kaggle_predictions = np.maximum(kaggle_predictions, 0)

submission = pd.DataFrame({
    "Id": Ids,
    "SalePrice":    kaggle_predictions
})

submission["SalePrice"].astype(float)

submission.to_csv("submission.csv", index=False)

print(submission.shape)                
print(submission.head())

(1459, 2)
     Id     SalePrice
0  1461  134475.40625
1  1462  134475.40625
2  1463  185323.34375
3  1464  185323.34375
4  1465  243550.75000


In [34]:
print(submission["SalePrice"].min())
print(submission["SalePrice"].max())
print(submission["SalePrice"].describe())
print(len(submission["SalePrice"].unique()))

87635.5
669948.75
count      1459.000000
mean     176498.406250
std       67177.171875
min       87635.500000
25%      134475.406250
50%      185323.343750
75%      185323.343750
max      669948.750000
Name: SalePrice, dtype: float64
52
